In [0]:
-- ═══════════════════════════════════════════════════════════
-- DIM DATE
-- Generated independently — no source table needed
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.dim_date AS
SELECT
    CAST(DATE_FORMAT(d, 'yyyyMMdd') AS INT)     AS date_key,
    d                                           AS full_date,
    YEAR(d)                                     AS year,
    QUARTER(d)                                  AS quarter,
    MONTH(d)                                    AS month_num,
    DATE_FORMAT(d, 'MMMM')                      AS month_name,
    DAY(d)                                      AS day_of_month,
    DAYOFWEEK(d)                                AS day_of_week_num,
    DATE_FORMAT(d, 'EEEE')                      AS day_name,
    CASE WHEN DAYOFWEEK(d) IN (1,7) 
         THEN true ELSE false END               AS is_weekend,
    CONCAT('Q', QUARTER(d), '-', YEAR(d))       AS quarter_label,
    DATE_FORMAT(d, 'MMM-yyyy')                  AS month_year_label
FROM (
    SELECT EXPLODE(
        SEQUENCE(DATE('2010-01-01'), DATE('2026-12-31'), INTERVAL 1 DAY)
    ) AS d
);

In [0]:
-- ═══════════════════════════════════════════════════════════
-- DIM CUSTOMER
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.dim_customer AS
SELECT
    ROW_NUMBER() OVER (ORDER BY c.customer_id)  AS customer_key,
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.zip_code,
    -- enrich with geolocation
    g.latitude,
    g.longitude
FROM paymentdw.silver.customers c
LEFT JOIN paymentdw.silver.geolocation g
    ON c.zip_code = g.zip_code;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- DIM SELLER
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.dim_seller AS
SELECT
    ROW_NUMBER() OVER (ORDER BY seller_id)      AS seller_key,
    seller_id,
    seller_city,
    seller_state,
    zip_code
FROM paymentdw.silver.sellers;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- DIM PRODUCT
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.dim_product AS
SELECT
    ROW_NUMBER() OVER (ORDER BY product_id)     AS product_key,
    product_id,
    category_english                            AS category,
    category_portuguese,
    weight_g,
    length_cm,
    height_cm,
    width_cm,
    photos_qty
FROM paymentdw.silver.products;

-- ═══════════════════════════════════════════════════════════
-- DIM PAYMENT METHOD
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.dim_payment_method AS
SELECT
    ROW_NUMBER() OVER (ORDER BY payment_type)   AS payment_method_key,
    payment_type,
    CASE payment_type
        WHEN 'credit_card'  THEN 'Credit Card'
        WHEN 'boleto'       THEN 'Boleto (Bank Slip)'
        WHEN 'voucher'      THEN 'Voucher'
        WHEN 'debit_card'   THEN 'Debit Card'
        ELSE 'Other'
    END                                         AS payment_type_label,
    CASE payment_type
        WHEN 'credit_card'  THEN 'Card'
        WHEN 'debit_card'   THEN 'Card'
        WHEN 'boleto'       THEN 'Bank Transfer'
        WHEN 'voucher'      THEN 'Voucher'
        ELSE 'Other'
    END                                         AS payment_category
FROM (
    SELECT DISTINCT payment_type
    FROM paymentdw.silver.order_payments
    WHERE payment_type IS NOT NULL
);

In [0]:
-- ═══════════════════════════════════════════════════════════
-- DIM CARD
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.dim_card AS
SELECT
    card_id                                     AS card_key,
    card_id,
    client_id,
    card_brand,
    card_type,
    has_chip,
    num_cards_issued,
    credit_limit,
    is_limit_zero,
    acct_open_date,
    year_pin_last_changed,
    card_on_dark_web
FROM paymentdw.silver.cards;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- DIM LOCATION
-- Built from geolocation — one row per zip code
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.dim_location AS
SELECT
    ROW_NUMBER() OVER (ORDER BY zip_code)       AS location_key,
    zip_code,
    city,
    state,
    latitude,
    longitude
FROM paymentdw.silver.geolocation;

In [0]:
SELECT 'dim_date'           AS table_name, COUNT(*) AS row_count FROM paymentdw.gold.dim_date
UNION ALL SELECT 'dim_customer',            COUNT(*) FROM paymentdw.gold.dim_customer
UNION ALL SELECT 'dim_seller',              COUNT(*) FROM paymentdw.gold.dim_seller
UNION ALL SELECT 'dim_product',             COUNT(*) FROM paymentdw.gold.dim_product
UNION ALL SELECT 'dim_payment_method',      COUNT(*) FROM paymentdw.gold.dim_payment_method
UNION ALL SELECT 'dim_card',                COUNT(*) FROM paymentdw.gold.dim_card
UNION ALL SELECT 'dim_location',            COUNT(*) FROM paymentdw.gold.dim_location
ORDER BY table_name;

## Fact Tables

In [0]:
-- ═══════════════════════════════════════════════════════════
-- FACT ORDERS
-- Grain: one row per order line item
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE paymentdw.gold.fact_orders
USING DELTA
PARTITIONED BY (order_year, order_month)
AS
WITH order_reviews AS (
    -- one review per order, take highest score if multiple
    SELECT
        order_id,
        MAX(review_score)   AS review_score,
        COUNT(*)            AS review_count
    FROM paymentdw.silver.reviews
    GROUP BY order_id
),
order_payments_agg AS (
    -- aggregate payments to order level
    SELECT
        order_id,
        SUM(payment_value)                      AS total_payment_value,
        MAX(payment_installments)               AS max_installments,
        COUNT(DISTINCT payment_type)            AS num_payment_methods,
        MAX(CASE WHEN is_refund THEN 1 ELSE 0 END) AS has_refund
    FROM paymentdw.silver.order_payments
    GROUP BY order_id
)
SELECT
    -- surrogate key
    ROW_NUMBER() OVER (ORDER BY oi.order_id, oi.order_item_id) AS order_item_key,
    -- foreign keys to dims
    o.order_id,
    oi.order_item_id,
    dc.customer_key,
    dp.product_key,
    ds.seller_key,
    dpm.payment_method_key,
    dl.location_key,
    CAST(DATE_FORMAT(o.purchase_timestamp, 'yyyyMMdd') AS INT)  AS date_key,
    -- date parts for partitioning
    YEAR(o.purchase_timestamp)                                  AS order_year,
    MONTH(o.purchase_timestamp)                                 AS order_month,
    -- order status
    o.order_status,
    o.is_delivered_on_time,
    o.delivery_days,
    -- measures
    CAST(oi.price AS DECIMAL(12,2))                             AS item_price,
    CAST(oi.freight_value AS DECIMAL(12,2))                     AS freight_value,
    CAST(oi.total_item_value AS DECIMAL(12,2))                  AS total_item_value,
    CAST(opa.total_payment_value AS DECIMAL(12,2))              AS total_payment_value,
    opa.max_installments,
    opa.num_payment_methods,
    CAST(opa.has_refund AS BOOLEAN)                             AS has_refund,
    -- review metrics
    orv.review_score,
    orv.review_count
FROM paymentdw.silver.order_items oi
-- join orders
JOIN paymentdw.silver.orders o
    ON oi.order_id = o.order_id
-- join dims
LEFT JOIN paymentdw.gold.dim_customer dc
    ON o.customer_id = dc.customer_id
LEFT JOIN paymentdw.gold.dim_product dp
    ON oi.product_id = dp.product_id
LEFT JOIN paymentdw.gold.dim_seller ds
    ON oi.seller_id = ds.seller_id
LEFT JOIN paymentdw.gold.dim_location dl
    ON dc.zip_code = dl.zip_code
-- join payment method dim on most common payment type per order
LEFT JOIN (
    SELECT order_id, payment_type
    FROM (
        SELECT order_id, payment_type,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY payment_value DESC
            ) AS rn
        FROM paymentdw.silver.order_payments
        WHERE payment_type IS NOT NULL
    ) WHERE rn = 1
) top_payment ON o.order_id = top_payment.order_id
LEFT JOIN paymentdw.gold.dim_payment_method dpm
    ON top_payment.payment_type = dpm.payment_type
-- join aggregated payment info
LEFT JOIN order_payments_agg opa
    ON o.order_id = opa.order_id
-- join reviews
LEFT JOIN order_reviews orv
    ON o.order_id = orv.order_id
WHERE o.purchase_timestamp IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE paymentdw.gold.fact_transactions
USING DELTA
PARTITIONED BY (txn_year, txn_month)
AS
SELECT
    t.transaction_id,
    t.client_id,                -- carry this directly from silver
    dca.card_key,
    dl.location_key,
    CAST(DATE_FORMAT(t.transaction_date, 'yyyyMMdd') AS INT)    AS date_key,
    YEAR(t.transaction_date)                                    AS txn_year,
    MONTH(t.transaction_date)                                   AS txn_month,
    t.merchant_id,
    t.merchant_city,
    t.merchant_state,
    t.mcc_code,
    t.use_chip                                                  AS payment_channel,
    CAST(t.amount AS DECIMAL(12,2))                             AS amount,
    t.is_debit,
    t.error_type,
    t.has_error,
    dca.card_on_dark_web,
    dca.is_limit_zero,
    dca.credit_limit,
    dca.card_brand,
    dca.card_type
FROM paymentdw.silver.transactions t
LEFT JOIN paymentdw.gold.dim_card dca
    ON t.card_id = dca.card_id
LEFT JOIN paymentdw.gold.dim_location dl
    ON t.zip = dl.zip_code
WHERE t.transaction_date IS NOT NULL;

In [0]:
SELECT 'fact_orders'        AS table_name, COUNT(*) AS row_count FROM paymentdw.gold.fact_orders
UNION ALL SELECT 'fact_transactions',       COUNT(*) FROM paymentdw.gold.fact_transactions;

In [0]:
INSERT INTO paymentdw.logs.pipeline_logs
    (layer, table_name, source_table, rows_read, rows_written, status, error_message, run_timestamp, duration_seconds)
VALUES
    ('gold', 'dim_date',            'generated',                    6209,       6209,       'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'dim_customer',        'silver.customers+geolocation', 99441,      99441,      'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'dim_seller',          'silver.sellers',               3095,       3095,       'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'dim_product',         'silver.products',              32951,      32951,      'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'dim_payment_method',  'silver.order_payments',        5,          5,          'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'dim_card',            'silver.cards',                 6146,       6146,       'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'dim_location',        'silver.geolocation',           19015,      19015,      'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'fact_orders',         'silver.order_items+orders',    112650,     112650,     'SUCCESS', null, current_timestamp(), 0),
    ('gold', 'fact_transactions',   'silver.transactions',          13305915,   13305915,   'SUCCESS', null, current_timestamp(), 0);